In [ ]:
2+2

In [ ]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from shapely.geometry import Point
from geopy.distance import geodesic

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

In [ ]:
event_path = r"C:\Users\teaching\Downloads\outage-recovery-forecasting\data_transients\event_level_dataset_clean.csv"

event_df = pd.read_csv(event_path)

event_df["CountyFIPS"] = event_df["CountyFIPS"].astype(str).str.zfill(5)

# event_start was not saved explicitly, so reconstruct it from event_id.
# event_id looks like: "12005_2018-10-10 17:00:00"
event_df["event_start"] = pd.to_datetime(event_df["event_id"].str.split("_").str[1])

print(event_df.shape)
print(event_df.columns.tolist())
print(event_df.head())

In [ ]:
ib_path = r"C:\Users\teaching\Downloads\outage-recovery-forecasting\data_raw\ibtracs_conus_subset.csv"

ib = pd.read_csv(ib_path, low_memory=False)

# Keep useful columns.
ib = ib[["SID", "ISO_TIME", "LAT", "LON", "USA_WIND", "NAME"]].copy()

# Clean time and numeric values.
ib["ISO_TIME"] = pd.to_datetime(ib["ISO_TIME"], errors="coerce")
ib = ib.dropna(subset=["ISO_TIME"])

ib = ib.rename(columns={
    "SID": "storm_id_ib",
    "LAT": "lat",
    "LON": "lon",
    "USA_WIND": "wind_knots",
    "NAME": "storm_name"
})

ib["lat"] = pd.to_numeric(ib["lat"], errors="coerce")
ib["lon"] = pd.to_numeric(ib["lon"], errors="coerce")
ib["wind_knots"] = pd.to_numeric(ib["wind_knots"], errors="coerce")
ib["wind_mps"] = ib["wind_knots"] * 0.514444
ib["year"] = ib["ISO_TIME"].dt.year

ib = ib.dropna(subset=["lat", "lon"])

print("IBTrACS rows:", len(ib))
print("IBTrACS storms:", ib["storm_id_ib"].nunique())
print(ib.head())

In [ ]:
# County geometries
url = "https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json"
counties = gpd.read_file(url)
counties = counties.rename(columns={"id": "CountyFIPS"})
counties["CountyFIPS"] = counties["CountyFIPS"].astype(str).str.zfill(5)

# State FIPS codes to exclude from contiguous USA.
# 02 = Alaska, 15 = Hawaii, 72 = Puerto Rico.
# Also exclude territories if present.
exclude_state_fips = {"02", "15", "60", "66", "69", "72", "78"}

counties["state_fips"] = counties["CountyFIPS"].str[:2]

conus_counties = counties[~counties["state_fips"].isin(exclude_state_fips)].copy()

# Project to an equal-area USA CRS for buffering and spatial operations.
conus_counties_5070 = conus_counties.to_crs("EPSG:5070")
conus_geom_5070 = conus_counties_5070.unary_union

# Florida centroids for event counties
fl_counties = counties[counties["CountyFIPS"].str.startswith("12")].copy()
fl_counties["centroid"] = fl_counties.geometry.centroid

print("CONUS counties:", len(conus_counties))
print("Florida counties:", len(fl_counties))

In [ ]:
MATCH_WINDOW_HOURS = 48

distances = []
matched_storms = []

for _, row in event_df.iterrows():
    event_time = row["event_start"]
    county = row["CountyFIPS"]

    # County centroid
    county_match = fl_counties[fl_counties["CountyFIPS"] == county]
    if county_match.empty:
        distances.append(np.nan)
        matched_storms.append(None)
        continue

    centroid = county_match["centroid"].iloc[0]
    county_latlon = (centroid.y, centroid.x)

    # Candidate IBTrACS points near event time
    window = ib[
        (ib["ISO_TIME"] >= event_time - pd.Timedelta(hours=MATCH_WINDOW_HOURS)) &
        (ib["ISO_TIME"] <= event_time + pd.Timedelta(hours=MATCH_WINDOW_HOURS))
    ].copy()

    if window.empty:
        distances.append(np.nan)
        matched_storms.append(None)
        continue

    dists = []
    storm_ids = []

    for _, r in window.iterrows():
        d = geodesic(county_latlon, (r["lat"], r["lon"])).km
        dists.append(d)
        storm_ids.append(r["storm_id_ib"])

    min_idx = int(np.argmin(dists))

    distances.append(dists[min_idx])
    matched_storms.append(storm_ids[min_idx])

event_df["min_dist_to_storm_km"] = distances
event_df["matched_ibtracs_storm"] = matched_storms

print("Distance summary:")
print(event_df["min_dist_to_storm_km"].describe())

print("\nMatched IBTrACS storms:")
print(event_df["matched_ibtracs_storm"].value_counts(dropna=False))

In [ ]:
CONUS_BUFFER_KM = 250  # captures storms close enough to affect land while centre is offshore
CONUS_BUFFER_M = CONUS_BUFFER_KM * 1000

# Create GeoDataFrame of IBTrACS points
ib_gdf = gpd.GeoDataFrame(
    ib.copy(),
    geometry=gpd.points_from_xy(ib["lon"], ib["lat"]),
    crs="EPSG:4326"
).to_crs("EPSG:5070")

# Strict: centre inside CONUS land polygon
ib_gdf["center_in_conus"] = ib_gdf.geometry.within(conus_geom_5070)

# Buffered: centre within N km of CONUS land polygon
conus_buffer_5070 = conus_geom_5070.buffer(CONUS_BUFFER_M)
ib_gdf["near_conus"] = ib_gdf.geometry.within(conus_buffer_5070)

# Compute storm max wind for both definitions.
strict_max = (
    ib_gdf[ib_gdf["center_in_conus"]]
    .groupby("storm_id_ib")["wind_mps"]
    .max()
    .reset_index()
    .rename(columns={"wind_mps": "storm_max_wind_center_in_conus_mps"})
)

near_max = (
    ib_gdf[ib_gdf["near_conus"]]
    .groupby("storm_id_ib")["wind_mps"]
    .max()
    .reset_index()
    .rename(columns={"wind_mps": "storm_max_wind_near_conus_mps"})
)

event_df = event_df.merge(
    strict_max,
    left_on="matched_ibtracs_storm",
    right_on="storm_id_ib",
    how="left"
).drop(columns=["storm_id_ib"], errors="ignore")

event_df = event_df.merge(
    near_max,
    left_on="matched_ibtracs_storm",
    right_on="storm_id_ib",
    how="left"
).drop(columns=["storm_id_ib"], errors="ignore")

print(event_df[[
    "storm",
    "matched_ibtracs_storm",
    "storm_max_wind_center_in_conus_mps",
    "storm_max_wind_near_conus_mps"
]].drop_duplicates().sort_values("storm"))

print("\nMissing strict CONUS max wind:", event_df["storm_max_wind_center_in_conus_mps"].isna().sum())
print("Missing near-CONUS max wind:", event_df["storm_max_wind_near_conus_mps"].isna().sum())

In [ ]:
save_path = r"C:\Users\teaching\Downloads\outage-recovery-forecasting\data_transients\event_level_dataset_rf_v3.csv"

event_df.to_csv(save_path, index=False)

print(f"Saved RF_v3 dataset to: {save_path}")
print(event_df.shape)

In [ ]:
# RF_v3 feature set.
# NOTE: max_customers_tracked is intentionally excluded because customersTracked is noisy / unreliable.
# CountyFIPS is tested separately as a categorical variable.

features_v3 = [
    "max_gust",
    "mean_gust_7d",
    "total_precip_7d",
    "pressure_min_7d",
    "county_pop",
    "min_dist_to_storm_km",
    "storm_max_wind_near_conus_mps"
]

target_col = "t90"
group_col = "storm"

X_nc = event_df[features_v3].copy()
X_wc = event_df[features_v3 + ["CountyFIPS"]].copy()
y = event_df[target_col].copy()
groups = event_df[group_col].copy()

print(X_nc.isna().sum())

In [ ]:
# No CountyFIPS
pre_nc = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), features_v3)
])

model_nc = Pipeline([
    ("preprocess", pre_nc),
    ("rf", RandomForestRegressor(
        n_estimators=200,
        max_depth=None,
        min_samples_leaf=3,
        random_state=42
    ))
])

# With CountyFIPS
pre_wc = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), features_v3),
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["CountyFIPS"])
])

model_wc = Pipeline([
    ("preprocess", pre_wc),
    ("rf", RandomForestRegressor(
        n_estimators=200,
        max_depth=None,
        min_samples_leaf=3,
        random_state=42
    ))
])

In [ ]:
logo = LeaveOneGroupOut()

def evaluate_logo_global(model, X, y, groups):
    preds = np.full(len(y), np.nan)
    fold_rows = []

    for i, (train_idx, test_idx) in enumerate(logo.split(X, y, groups), start=1):
        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]
        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        test_storm = groups.iloc[test_idx].iloc[0]

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        preds[test_idx] = y_pred

        fold_rows.append({
            "fold": i,
            "storm": test_storm,
            "n": len(test_idx),
            "mae_diagnostic_per_storm": mean_absolute_error(y_test, y_pred),
            "rmse_diagnostic_per_storm": np.sqrt(mean_squared_error(y_test, y_pred)),
        })

    fold_df = pd.DataFrame(fold_rows)

    global_mae = mean_absolute_error(y, preds)
    global_rmse = np.sqrt(mean_squared_error(y, preds))

    return preds, fold_df, global_mae, global_rmse

pred_nc, fold_nc, mae_nc, rmse_nc = evaluate_logo_global(model_nc, X_nc, y, groups)
pred_wc, fold_wc, mae_wc, rmse_wc = evaluate_logo_global(model_wc, X_wc, y, groups)

print("RF_v3 no CountyFIPS:")
print("Global MAE:", mae_nc)
print("Global RMSE:", rmse_nc)

print("\nRF_v3 with CountyFIPS:")
print("Global MAE:", mae_wc)
print("Global RMSE:", rmse_wc)

print("\nPer-storm diagnostic errors, no CountyFIPS:")
print(fold_nc.sort_values("mae_diagnostic_per_storm", ascending=False))

In [ ]:
def plot_scatter_by_storm(preds, title):
    plt.figure(figsize=(7, 6))

    for storm in event_df["storm"].unique():
        s = event_df["storm"] == storm
        plt.scatter(
            event_df.loc[s, "t90"],
            preds[s],
            label=storm,
            alpha=0.8
        )

    m = max(event_df["t90"].max(), np.nanmax(preds))
    plt.plot([0, m], [0, m], "--")

    plt.xlabel("Actual t90")
    plt.ylabel("Predicted t90")
    plt.title(title)
    plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

plot_scatter_by_storm(pred_nc, "RF_v3 no CountyFIPS")
plot_scatter_by_storm(pred_wc, "RF_v3 with CountyFIPS")

In [ ]:
rf_model_nc = model_nc.named_steps["rf"]

importances = rf_model_nc.feature_importances_
feat_imp = pd.Series(importances, index=features_v3).sort_values(ascending=False)

print(feat_imp)

feat_imp.plot(kind="barh")
plt.gca().invert_yaxis()
plt.title("RF_v3 Feature Importance — no CountyFIPS")
plt.show()

In [ ]:
corr_cols = features_v3 + ["t90"]

print(event_df[corr_cols].corr(numeric_only=True)["t90"].sort_values(ascending=False))